well in this first excerpt we're importing a kaggle dataset on indian classical ragas.
So in this dataset only I'll be implementing backprop 

BELOW CODE IS MANUAL BACKPROP IN NEURAL NETS USING TANH AS AN ACTIVATION FUNCTION

In [12]:
class Value:
    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def __pow__(self, k):
        assert isinstance(k, (int, float)), "only int/float powers"
        out = Value(self.data ** k, (self,), f'**{k}')
        def _backward():
            self.grad += k * self.data ** (k - 1) * out.grad
        out._backward = _backward
        return out

    def exp(self):
        e = math.exp(self.data)
        out = Value(e, (self,), 'exp')
        def _backward():
            self.grad += e * out.grad
        out._backward = _backward
        return out

    def log(self):
        out = Value(math.log(self.data), (self,), 'log')
        def _backward():
            self.grad += (1.0 / self.data) * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t, (self,), 'tanh')
        def _backward():
            self.grad += (1 - t ** 2) * out.grad
        out._backward = _backward
        return out

    def relu(self):
        out = Value(max(0.0, self.data), (self,), 'relu')
        def _backward():
            self.grad += (out.data > 0) * out.grad
        out._backward = _backward
        return out

    def backward(self):
        topo, visited = [], set()
        def build(v):
            if v not in visited:
                visited.add(v)
                for c in v._prev:
                    build(c)
                topo.append(v)
        build(self)
        self.grad = 1.0
        for v in reversed(topo):
            v._backward()

    def __neg__(self):             return self * -1
    def __sub__(self, other):      return self + (-other)
    def __rsub__(self, other):     return other + (-self)
    def __radd__(self, other):     return self + other
    def __rmul__(self, other):     return self * other
    def __truediv__(self, other):  return self * other ** -1
    def __rtruediv__(self, other): return other * self ** -1


class Module:
    def zero_grad(self):
        for p in self.parameters():
            p.grad = 0.0

    def parameters(self):        return []


class Neuron(Module):
    def __init__(self, nin, nonlin=True):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(0.0)
        self.nonlin = nonlin

    def __call__(self, x):
        act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        return act.tanh() if self.nonlin else act

    def parameters(self):
        return self.w + [self.b]


class Layer(Module):
    def __init__(self, nin, nout, **kw):
        self.neurons = [Neuron(nin, **kw) for _ in range(nout)]

    def __call__(self, x):
        out = [n(x) for n in self.neurons]
        return out[0] if len(out) == 1 else out

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]


class MLP(Module):
    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i + 1], nonlin=(i != len(nouts) - 1))
                       for i in range(len(nouts))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for l in self.layers for p in l.parameters()]

here in this excerpt we'll be importing the data from the raga datset using kagglehub library as the dataset is from one of kaggle. We basically built a path fucntion which would create dataframes into vectors.
(we know why we need vectors in nn's...)

In [ ]:

import os, sys, glob, io, contextlib, random, math
import numpy as np, pandas as pd


def get_path():
    if len(sys.argv) > 1 and os.path.isdir(sys.argv[1]):
        return sys.argv[1]
    import kagglehub
    with contextlib.redirect_stdout(io.StringIO()):
        return kagglehub.dataset_download(
            "anirudhsub/indian-art-music-raga-recognition-dataset")


def build_index(path):
    rows = []
    for p in glob.glob(os.path.join(path, "*", "*", "*---pitches.txt")):
        rows.append({"raga": p.split(os.sep)[-3],
                     "pitches": p,
                     "confidences": p.replace("pitches", "confidences"),
                     "tonic": p.replace("pitches", "tonic")})
    return pd.DataFrame(rows)


def pitch_hist(pitch_file, tonic_file, bins):
    f = np.loadtxt(pitch_file)
    tonic = float(np.atleast_1d(np.loadtxt(tonic_file))[0])
    f = f[f > 0]                                    # drop unvoiced frames
    cents = (1200 * np.log2(f / tonic)) % 1200      # fold into one octave
    h, _ = np.histogram(cents, bins=bins, range=(0, 1200))
    return h / h.sum()





training loop


In [14]:
def softmax_ce(logits, y):
    m = max(l.data for l in logits)                 # constant, for stability
    exps = [(l - m).exp() for l in logits]
    s = sum(exps)
    return s.log() - (logits[y] - m)                # -log softmax[y]


def accuracy(model, X, y):
    hits = 0
    for xi, yi in zip(X, y):
        logits = model(xi)
        hits += int(np.argmax([l.data for l in logits]) == yi)
    return hits / len(y)


CFG = dict(K=3, PER_RAGA=40, BINS=24, HIDDEN=[8], EPOCHS=60, LR=0.1, SEED=0)


def train(index, cfg):
    random.seed(cfg["SEED"])
    np.random.seed(cfg["SEED"])

    top = list(index.raga.value_counts().index[:cfg["K"]])
    n_classes = len(top)
    parts = [index[index.raga == r].sample(min((index.raga == r).sum(), cfg["PER_RAGA"]),
                                           random_state=cfg["SEED"]) for r in top]
    sub = pd.concat(parts).sample(frac=1, random_state=cfg["SEED"])
    print("using ragas:", top, "| recordings:", len(sub))

    X = np.array([pitch_hist(r.pitches, r.tonic, cfg["BINS"]) for r in sub.itertuples()])
    y = [top.index(r) for r in sub.raga]

    n_train = int(0.8 * len(X))
    Xtr, Xte, ytr, yte = X[:n_train], X[n_train:], y[:n_train], y[n_train:]

    mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-8         # standardize with train stats only
    Xtr = ((Xtr - mu) / sd).tolist()
    Xte = ((Xte - mu) / sd).tolist()

    model = MLP(cfg["BINS"], list(cfg["HIDDEN"]) + [n_classes])
    print("params:", len(model.parameters()), "\n")

    for epoch in range(cfg["EPOCHS"]):
        loss = sum(softmax_ce(model(xi), yi) for xi, yi in zip(Xtr, ytr)) * (1 / len(Xtr))
        model.zero_grad()
        loss.backward()
        for p in model.parameters():
            p.data -= cfg["LR"] * p.grad
        if epoch % 5 == 0 or epoch == cfg["EPOCHS"] - 1:
            print(f"epoch {epoch:3d}  loss {loss.data:.4f}  "
                  f"train acc {accuracy(model, Xtr, ytr):.2f}  "
                  f"test acc {accuracy(model, Xte, yte):.2f}")
    return accuracy(model, Xte, yte)


if __name__ == "__main__":
    cfg = {**CFG, "K": 5, "BINS": 36}               # per-run overrides

    path = get_path()
    index = build_index(path)
    print("index:", index.shape)
    if index.empty:
        raise SystemExit(f"No *---pitches.txt files found under {path}; check the folder layout.")
    print(index.raga.value_counts().head(10), "\n")

    train(index, cfg)

Using Colab cache for faster access to the 'indian-art-music-raga-recognition-dataset' dataset.
index: (480, 4)
raga
Kalyāṇi              12
Karaharapriya          12
Kāmavardani           12
Sencuruṭṭi           12
Harikāmbhōji         12
Bhairavi               12
Śrī                  12
Sahānā               12
Gauḷa                 12
Yadukula kāṁbōji    12
Name: count, dtype: int64 

using ragas: ['Kalyāṇi', 'Karaharapriya', 'Kāmavardani', 'Sencuruṭṭi', 'Harikāmbhōji'] | recordings: 60
params: 341 

epoch   0  loss 2.4499  train acc 0.23  test acc 0.17
epoch   5  loss 1.6899  train acc 0.46  test acc 0.25
epoch  10  loss 1.2383  train acc 0.56  test acc 0.25
epoch  15  loss 0.9641  train acc 0.67  test acc 0.42
epoch  20  loss 0.7906  train acc 0.71  test acc 0.42
epoch  25  loss 0.6721  train acc 0.83  test acc 0.42
epoch  30  loss 0.5849  train acc 0.88  test acc 0.50
epoch  35  loss 0.5176  train acc 0.94  test acc 0.50
epoch  40  loss 0.4645  train acc 0.94